# Underwater with SE(3)

<a href="https://colab.research.google.com/github/gtbook/robotics/blob/main/S81_underwater_state.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    %pip install --quiet gtsam-develop otter-grader
except ImportError:
    pass

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import gtsam
from typing import Tuple
import otter
grader = otter.Notebook()

```{index} state; 3D pose space
```

> Motion in 3D has 6 degrees of freedom.

## Instructions for Running and Submission

### Local
Ensure you have the notebook (S81_underwater_state.ipynb) and the tests folder in the same directory, and that you are currently in that directory.

In your environment (Python or Conda), install gtsam-develop (pip install gtsam-develop) and otter-grader (pip install otter-grader). **Note**: gtsam-develop is only available on macOS/Linux, so if you're on a Windows, you can either use WSL2 or Colab. If you aren't already familiar with WSL2, Colab will be quickest.

### Colab
Upload the notebook to Colab and then upload the .py files in the tests folder to another tests folder (must be same name) that you create in your Colab files. You are unable to upload folders to Colab, so you need to place the files in a created folder manually.

The required packages will be installed in the first cell above.

### Submission
Submit this notebook with all of your work to the Gradescope assignment.

## Assignment

We can calculate the exponential map $\exp(\hat{\xi})$ using two common definitions extended to matrices:

1. **The Limit Definition**: 
   Based on the compound interest formula $(1 + \frac{x}{n})^n$, for matrices this becomes:
   $$ \exp(\hat{\xi}) = \lim_{n \to \infty} \left(I + \frac{1}{n}\hat{\xi}\right)^n $$

2. **The Power Series Expansion**: 
   Just as $e^x = \sum \frac{x^n}{n!}$ for scalars, for the matrix $\hat{\xi}$ we have:
   $$ \exp(\hat{\xi}) = I + \hat{\xi} + \frac{1}{2!} \hat{\xi}^2 + \frac{1}{3!} \hat{\xi}^3 + \dots $$

Both converge to the same result, which is the analytic exponential map provided by `gtsam`.

In [ ]:
import numpy as np

# Define a random twist for Pose2 (omega, v_x, v_y)
xi = np.array([0.1, 0.2, 0.3])

# Compute Ground Truth using GTSAM
T_exact = gtsam.Pose2.Expmap(xi).matrix()
print(f"Exact:\n{np.round(T_exact, 4)}")

# Get the matrix representation of the twist
Xi = gtsam.Pose2.Hat(xi)
print(f"Xi:\n{np.round(Xi, 4)}")

Exact:
[[ 0.9553 -0.2955  0.0687]
 [ 0.2955  0.9553  0.2119]
 [ 0.      0.      1.    ]]
Xi:
[[ 0.  -0.3  0.1]
 [ 0.3  0.   0.2]
 [ 0.   0.   0. ]]


In [4]:
# Define nr iterations
n = 10000

# Approximation 1: Limit definition
T_limit = np.linalg.matrix_power(np.eye(3) + Xi/n, n)
print(f"Limit Approx:\n{np.round(T_limit, 4)}")
error_limit = np.linalg.norm(T_exact - T_limit, 'fro')
print(f"Error Limit:  {error_limit:.2e}")

Limit Approx:
[[ 0.9553 -0.2955  0.0687]
 [ 0.2955  0.9553  0.2119]
 [ 0.      0.      1.    ]]
Error Limit:  7.19e-06


**Question 1 (5 points)**: Write the Power Series approximation for the exponential map and compute the Frobenius norm of the difference between it and the ground truth.

In [11]:
# Implementation copied over from project_1.ipynb 

def power_series(Xi: np.ndarray) -> Tuple[np.ndarray, np.float32]:
    n = 20  # number of terms in the series expansion
    I = np.identity(3)  # identity matrix
    
    result = I.copy()
    hat_power = I.copy()  # to hold powers of the hat matrix
    
    for i in range(1, n + 1):
        hat_power = hat_power @ Xi / i  # compute Xi^i / i!
        result += hat_power
    
    T_series = result
    error_series = np.linalg.norm(T_exact - T_series, 'fro')
    return T_series, error_series

T_series, error_series = power_series(Xi=Xi)
print(f'Power Series Approx:\n{T_series}')
print(f'Error Series:\n{error_series}')

Power Series Approx:
[[ 0.95533649 -0.29552021  0.06873106]
 [ 0.29552021  0.95533649  0.21190131]
 [ 0.          0.          1.        ]]
Error Series:
6.938893903907228e-17


In [12]:
grader.check("q1")

q1 results: All test cases passed!

Look at how much better power series is with only 20 terms !

In addition, we will go beyond the drone chapter:
- Poses versus transformations. Torsor (poses, manifold only) vs. transforms (SE(3), the group)
- What is the definition of the Jacobian of a mapping between two manifolds? 
- A simple mapping is the inverse. What is its Jacobian? 
- What about acting on a point? Now it's two Jacobians.
- The most important mapping for this chapter, composition of two rotations or two transforms. 

Exercise:
- Derive the Jacobian compose

### Jacobian of Inverse
$$
\begin{aligned}
T^{-1}(I + \hat{\zeta}) &\approx (T(I+\hat{\xi}))^{-1} \approx (I-\hat{\xi})T^{-1} \\
T^{-1}\hat{\zeta} &\approx - \hat{\xi}T^{-1} \\
\hat{\zeta} &\approx -T\hat{\xi}T^{-1}
\end{aligned}
$$
But what do we do with the result? Can we write it as $H \xi$ ? Yes:
$$
\zeta = -\text{Ad}_T \xi
$$

### The Adjoint Map: Transforming Twists

We often need to transform a velocity vector (a twist) from one coordinate frame to another. 

To understand how a twist $\xi$ changes when we transform it by a pose $T$, let's look at how we transform a **small movement**.

Recall from the previous exercise that a pose representing a small movement can be approximated as:
$$ T_{\Delta} \approx I + \hat{\xi} $$

If we change the coordinate frame of this small movement using a transformation $T$, the standard formula for changing the basis of a transformation matrix is $T_{\Delta}' = T T_{\Delta} T^{-1}$. Let's see what happens to our approximation:

$$
\begin{aligned}
T_{\Delta}' &= T (I + \hat{\xi}) T^{-1} \\
&= T I T^{-1} + T \hat{\xi} T^{-1} \\
&= I + T \hat{\xi} T^{-1}
\end{aligned}
$$

We know the result $T_{\Delta}'$ must also be a small movement corresponding to some new twist $\xi'$. Therefore, comparing the result to $I + \hat{\xi}'$, we find a direct relationship between the twist matrices:

$$ \hat{\xi}' = T \hat{\xi} T^{-1} $$

The **Adjoint Map** ($\text{Ad}_T$) is simply the linear function that extracts the vector $\xi'$ from this matrix calculation.


### Derivation of Adjoint Map for SE(n)

We derive $\text{Ad}_T$ by explicitly computing the conjugation $\hat{\xi}' = T \hat{\xi} T^{-1}$ using block matrices.

Given $T = \begin{bmatrix} R & t \\ 0 & 1 \end{bmatrix}$ and $\hat{\xi} = \begin{bmatrix} \Omega & v \\ 0 & 0 \end{bmatrix}$, the inverse is $T^{-1} = \begin{bmatrix} R^T & -R^T t \\ 0 & 1 \end{bmatrix}$.

1.  **Multiply $T$ and $\hat{\xi}$:**
    $$ T \hat{\xi} = \begin{bmatrix} R & t \\ 0 & 1 \end{bmatrix} \begin{bmatrix} \Omega & v \\ 0 & 0 \end{bmatrix} = \begin{bmatrix} R\Omega & Rv \\ 0 & 0 \end{bmatrix} $$

2.  **Multiply result by $T^{-1}$:**
    $$
    \begin{aligned}
    \hat{\xi}' &= \begin{bmatrix} R\Omega & Rv \\ 0 & 0 \end{bmatrix} \begin{bmatrix} R^T & -R^T t \\ 0 & 1 \end{bmatrix} \\
    &= \begin{bmatrix} R\Omega R^T & -R\Omega R^T t + Rv \\ 0 & 0 \end{bmatrix}
    \end{aligned}
    $$

3.  **Identify components of $\xi' = [\omega', v']^T$:**
    *   **Angular:** $\Omega' = R \Omega R^T$. By property of skew-symmetric matrices, this implies $\omega' = R\omega$.
    *   **Linear:** $v' = Rv - \Omega' t$. Using the cross product property $-\Omega' t = -(\omega' \times t) = t \times \omega' = [t]_\times R\omega$.
    $$ v' = [t]_\times R\omega + Rv $$

4.  **Assemble the Adjoint matrix:**
    $$ \begin{bmatrix} \omega' \\ v' \end{bmatrix} = \begin{bmatrix} R & 0 \\ [t]_\times R & R \end{bmatrix} \begin{bmatrix} \omega \\ v \end{bmatrix} \implies \text{Ad}_T = \begin{bmatrix} R & 0 \\ [t]_\times R & R \end{bmatrix} $$

### Jacobian of Compose
$$
\begin{aligned}
T_1 T_2 (I + \hat{\zeta}) &\approx T_1(I+\hat{\xi}_1) T_2(I+\hat{\xi}_2) \approx T_1 T_2 + T_1 \hat{\xi}_1 T_2 + T_1 T_2 \hat{\xi}_2 \\
T_1 T_2 \hat{\zeta} &\approx T_1 \hat{\xi}_1 T_2 + T_1 T_2 \hat{\xi}_2 \\
\hat{\zeta} &\approx (T_1 T_2)^{-1} (T_1 \hat{\xi}_1 T_2) + \hat{\xi}_2 = T_2^{-1} \hat{\xi}_1 T_2 + \hat{\xi}_2 \\
\zeta &= \text{Ad}_{T_2^{-1}} \xi_1 + \xi_2
\end{aligned}
$$

This gives us **two** Jacobians:
1. With respect to $T_1$: $\text{Ad}_{T_2^{-1}}$
2. With respect to $T_2$: Identity ($I$)

**Question 2 (5 points)**: Compute the Jacobians of Compose and compare to the Analytic Adjoint of T2.inverse().

In [ ]:
# Create two random Pose2 objects
T1 = gtsam.Pose2(1, 2, 3)
T2 = gtsam.Pose2(4, 5, 6)

def compute_jacobians(T1: gtsam.Pose2, T2: gtsam.Pose2) -> Tuple[gtsam.Pose2, gtsam.Pose2, np.ndarray, np.ndarray, np.ndarray]:
    # Allocate memory for Jacobians (Fortran order is preferred by GTSAM wrappers internally)
    H1 = np.zeros((3, 3), order='F') # 'F' for fortran order which is column-major
    H2 = np.zeros((3, 3), order='F') 

    # Compute composition and retrieve Jacobians
    T_composed = T1.compose(T2, H1, H2)

    # Compute the Analytic Adjoint of T2.inverse()
    adj_T2inv = T2.inverse().AdjointMap()

    return H1, H2, adj_T2inv

H1, H2, adj_T2inv = compute_jacobians(T1, T2)

print(f'Jacobian w.r.t. T1:\n{H1}')
print(f'Jacobian w.r.t. T2:\n{H2}')

Jacobian w.r.t. T1:
[[ 0.96017029 -0.2794155  -5.91851343]
 [ 0.2794155   0.96017029  2.44360366]
 [ 0.          0.          1.        ]]
Jacobian w.r.t. T2:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


In [24]:
grader.check("q2")

q2 results: All test cases passed!

## Sources

- [A micro Lie theory for state estimation in robotics, Joan Solà, Jeremie Deray, Dinesh Atchuthan, 2018](https://arxiv.org/abs/1812.01537)
- [State estimation for Robotics, Tim Barfoot, 2025](https://www.cambridge.org/core/books/state-estimation-for-robotics/AC0E0AC229C55203B3C8F106BCB61F48), find [**pdf here**](http://asrl.utias.utoronto.ca/~tdb/bib/barfoot_ser24.pdf)
- [an introduction to Optimization on smooth manifolds, Nicolas Boumal, 2023](https://www.nicolasboumal.net/book/)
- Richard M. Murray, Shankar Shastry, Zexiang Li, S. Shankar Sastry, S. Shankara Sastry,
Mathematical Introduction to Robotic Manipulation, 1994